Notebook 3 - Convex Portfolio Optimization

In [10]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "data"

Since sample mean and sample covariance are rather bad ways to estimate expected return in such small time periods like a year, due the large amount of noise, we use the Ledoit-Wolf shrinkage estimator to estimate the covariance matrix.

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from sklearn.covariance import LedoitWolf

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (12, 5)
np.random.seed(42)

In [12]:
# Asset returns (simple returns for portfolio accounting)
simplert = pd.read_csv(DATA / 'processed' / 'simplert.csv', index_col=0, parse_dates=True)
print('Asset returns shape:', simplert.shape)
print('Assets:', simplert.columns.tolist())

# Regime probabilities from HMM
regimeprobs = pd.read_csv(DATA / 'processed' / 'regimeprobs.csv', index_col=0, parse_dates=True)
print('\nRegime probabilities shape:', regimeprobs.shape)
print('Columns:', regimeprobs.columns.tolist())

simplert.head()

Asset returns shape: (5032, 4)
Assets: ['SPY', 'TLT', 'GLD', 'CASH']

Regime probabilities shape: (4740, 5)
Columns: ['pcrisis', 'precovery', 'pbull', 'regime', 'regimename']


,SPY,TLT,GLD,CASH
Date,,,,
2005-01-04,-0.012219,-0.010480,-0.006509,NaN
2005-01-05,-0.006901,0.005352,-0.001638,0.000081
2005-01-06,0.005084,0.000679,-0.012186,0.000081
2005-01-07,-0.001433,0.002264,-0.007355,0.000081
2005-01-10,0.004728,0.001581,0.002629,0.000081


In [13]:
# The NaN value is due to the fact that it is the first day and the data was created with a lag of one day.
# We can drop this row, as otherwise the Ledoit-Wolf estimator will not work.
simplert = simplert.dropna()
print('After dropping NaN rows:', simplert.shape)


After dropping NaN rows: (5031, 4)


We implement linear programming to find the minimum variance portfolio and the maximum Sharpe ratio portfolio.

In [14]:
def minvarianceportfolio(sigma):
    """
    Find portfolio weights minimizing variance.
    
    Parameter(s):
    sigma : np.ndarray, shape (n, n)
        Covariance matrix of asset returns (annualized).

    Return(s):
    w : np.ndarray, shape (n,)
        Optimal weights, sum to 1, all >= 0.
    """
    n = sigma.shape[0]
    w = cp.Variable(n)
    
    objective = cp.Minimize(cp.quad_form(w, sigma))
    constraints = [
        cp.sum(w) == 1,   # fully invested
        w >= 0,           # long-only
    ]
    
    problem = cp.Problem(objective, constraints)
    problem.solve()
    
    if problem.status != 'optimal':
        raise RuntimeError(f'Solver failed: {problem.status}')
    
    return w.value

In [15]:
def maxsharpeportfolio(mu, sigma):
    """
    Find portfolio weights maximizing the Sharpe ratio using convex reformulation.
    
    Parameter(s)
    mu : np.ndarray, shape (n,)
        Expected returns (annualized).
    sigma : np.ndarray, shape (n, n)
        Covariance matrix (annualized).
    
    Return(s):
    w : np.ndarray, shape (n,)
        Optimal weights, sum to 1, all >= 0.
    """
    n = len(mu)
    
    # If all expected returns are <= 0, max-sharpe is undefined and we fall back to min variance.
    if np.max(mu) <= 0:
        return minvarianceportfolio(sigma)
    
    y = cp.Variable(n)
    
    objective = cp.Minimize(cp.quad_form(y, sigma))
    constraints = [
        mu @ y == 1,
        y >= 0,
    ]
    
    problem = cp.Problem(objective, constraints)
    problem.solve()
    
    if problem.status != 'optimal':
        # Fall back to min variance if the reformulation fails
        return minvarianceportfolio(sigma)
    
    # Recover weights by normalizing y to sum to 1
    w = y.value / np.sum(y.value)
    return w

In [16]:
def equalweightportfolio(nassets):
    return np.ones(nassets) / nassets

In [18]:
# As a sample run, we take the last 252 days of asset returns.

window = simplert.iloc[-252:]
assets = window.columns.tolist()
print('Window:', window.index[0].date(), 'to', window.index[-1].date())

mu = window.mean().values * 252       
Sigma_sample = window.cov().values * 252


lw = LedoitWolf().fit(window.values)
Sigma = lw.covariance_ * 252

print('\nAnnualized expected returns:')
for a, m in zip(assets, mu):
    print(f'  {a}: {m*100:+.1f}%')

print('\nAnnualized volatilities (sqrt of diagonal):')
for a, s in zip(assets, np.sqrt(np.diag(Sigma))):
    print(f'  {a}: {s*100:.1f}%')


wminvar = minvarianceportfolio(Sigma)
wmaxsr  = maxsharpeportfolio(mu, Sigma)
wew     = equalweightportfolio(len(assets))

results = pd.DataFrame({
    'Min Variance': wminvar,
    'Max Sharpe':   wmaxsr,
    'Equal Weight': wew,
}, index=assets).round(3)

print('\nOptimal portfolios:')
print(results)

Window: 2024-01-02 to 2024-12-31

Annualized expected returns:
  SPY: +23.0%
  TLT: -7.4%
  GLD: +24.8%
  CASH: +5.3%

Annualized volatilities (sqrt of diagonal):
  SPY: 12.5%
  TLT: 14.0%
  GLD: 14.8%
  CASH: 3.2%

Optimal portfolios:
      Min Variance  Max Sharpe  Equal Weight
SPY          0.049       0.169          0.25
TLT          0.039       0.000          0.25
GLD          0.024       0.122          0.25
CASH         0.888       0.709          0.25


In [ ]:
# Sanity Check
def portfolio_stats(w, mu, sigma):
    """Annualized return, vol, sharpe."""
    ret = w @ mu
    vol = np.sqrt(w @ sigma @ w)
    sharpe = ret / vol if vol > 0 else 0
    return ret, vol, sharpe

print('Portfolio statistics (annualized):')
print(f'{"Strategy":<15} {"Return":>10} {"Vol":>8} {"Sharpe":>11}')
for name, w in [('Min Variance', wminvar), ('Max Sharpe', wmaxsr), ('Equal Weight', wew)]:
    r, v, s = portfolio_stats(w, mu, Sigma)
    print(f'{name:<15} {r*100:>8.1f}% {v*100:>8.1f}% {s:>10.2f}')

Portfolio statistics (annualized):
Strategy            Return      Vol      Sharpe
Min Variance         6.1%      3.0%       2.02
Max Sharpe          10.6%      3.9%       2.76
Equal Weight        11.4%      7.0%       1.64
